# 🔗 MinIO S3 Data Explorer Silver

In [1]:
# Cài đặt các thư viện cần thiết
import subprocess
import sys

# Tất cả packages cần thiết cho notebook
packages = [
    'minio',
    'pandas',
    'pyarrow',
    'confluent-kafka',
    'fastavro',
    'cachetools',
    'httpx',
    'attrs',
    'authlib',
    'requests'
]

print("[INFO] Cài đặt thư viện...\n")
failed_packages = []

for package in packages:
    try:
        # Thử import package (handle package name khác import name)
        if package == 'confluent-kafka':
            __import__('confluent_kafka')
        elif package == 'pyarrow':
            __import__('pyarrow')
        elif package == 'httpx':
            __import__('httpx')
        elif package == 'cachetools':
            __import__('cachetools')
        else:
            __import__(package)
        print(f"[OK] {package:<25} - đã có sẵn")
    except ImportError:
        print(f"[INSTALL] Đang cài {package:<25}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
            print(f"[OK] {package:<25} - cài xong")
        except Exception as err:
            print(f"[ERROR] {package:<25} - LỖI: {err}")
            failed_packages.append(package)

if failed_packages:
    print(f"\n[WARN] Các package cài đặt thất bại: {failed_packages}")
else:
    print("\n[OK] Tất cả thư viện sẵn sàng!")

[INFO] Cài đặt thư viện...

[OK] minio                     - đã có sẵn
[OK] pandas                    - đã có sẵn
[OK] pyarrow                   - đã có sẵn
[OK] confluent-kafka           - đã có sẵn
[OK] fastavro                  - đã có sẵn
[OK] cachetools                - đã có sẵn
[OK] httpx                     - đã có sẵn
[OK] attrs                     - đã có sẵn
[OK] authlib                   - đã có sẵn
[OK] requests                  - đã có sẵn

[OK] Tất cả thư viện sẵn sàng!


In [3]:

# Import libraries
import os
import json
from minio import Minio
from minio.deleteobjects import DeleteObject
import pandas as pd
from datetime import datetime
import pyarrow.parquet as pq
from pathlib import Path

print("[INFO] Imports hoàn tất!")


[INFO] Imports hoàn tất!


## MinIO Configuration for Silver Layer Exploration

In [4]:

# MinIO Connection Configuration
MINIO_CONFIG = {
    'endpoint': 'localhost:9000',
    'access_key': 'minioadmin',
    'secret_key': 'minioadmin',
    'secure': False,
    'region': 'ap-southeast-1'
}

BUCKET_NAME = 'data-lake'
SILVER_PREFIX = 'silver/'

# Connect to MinIO
try:
    minio_client = Minio(
        MINIO_CONFIG['endpoint'],
        access_key=MINIO_CONFIG['access_key'],
        secret_key=MINIO_CONFIG['secret_key'],
        secure=MINIO_CONFIG['secure'],
        region=MINIO_CONFIG['region']
    )
    
    # Test connection
    buckets = minio_client.list_buckets()
    print("[SUCCESS] MinIO Connected!")
    print("[INFO] Available buckets: " + str(len(buckets)))
    for bucket in buckets:
        print("   - " + bucket.name)
    
except Exception as e:
    print("[ERROR] MinIO Connection failed: " + str(e))
    raise


[SUCCESS] MinIO Connected!
[INFO] Available buckets: 1
   - data-lake


## Silver Layer Tables Overview

Silver tables (5 total):
1. dim_customers (SCD2) - Customer master with history
2. dim_products (SCD1) - Product master (current only)
3. fact_orders - Order transactions
4. fact_payments - Payment transactions
5. fact_shipments - Shipment transactions

In [5]:

# List all Silver tables from MinIO
def list_silver_tables():
    """List all Silver Iceberg tables from MinIO"""
    try:
        objects = minio_client.list_objects(
            BUCKET_NAME,
            prefix="warehouse/silver/",
            recursive=True
        )
        
        # Organize by table name
        tables = {}
        for obj in objects:
            # Path format: warehouse/silver/table_name/metadata/v1/...
            parts = obj.object_name.split('/')
            if len(parts) >= 3:
                table_name = parts[2]
                if table_name not in tables:
                    tables[table_name] = {
                        'object_count': 0,
                        'total_size_mb': 0,
                        'file_types': {}
                    }
                tables[table_name]['object_count'] += 1
                tables[table_name]['total_size_mb'] += obj.size / (1024 * 1024)
                
                file_ext = obj.object_name.split('.')[-1] if '.' in obj.object_name else 'unknown'
                file_types_dict = tables[table_name]['file_types']
                file_types_dict[file_ext] = file_types_dict.get(file_ext, 0) + 1
        
        return tables
    except Exception as e:
        print("[ERROR] Error listing tables: " + str(e))
        return {}

silver_tables = list_silver_tables()
print("\n[INFO] Silver Tables in MinIO:")
print("=" * 70)

for table_name in sorted(silver_tables.keys()):
    table_info = silver_tables[table_name]
    print("[TABLE] " + table_name)
    print("  Objects: " + str(table_info['object_count']))
    print("  Total Size (MB): {:.2f}".format(table_info['total_size_mb']))
    print("  File Types: " + str(table_info['file_types']))
    print("-" * 70)

print("[INFO] Total Silver Tables: " + str(len(silver_tables)))



[INFO] Silver Tables in MinIO:
[TABLE] dim_customers
  Objects: 8
  Total Size (MB): 0.17
  File Types: {'parquet': 1, 'avro': 3, 'json': 3, 'text': 1}
----------------------------------------------------------------------
[TABLE] dim_products
  Objects: 8
  Total Size (MB): 0.11
  File Types: {'parquet': 1, 'avro': 3, 'json': 3, 'text': 1}
----------------------------------------------------------------------
[TABLE] fact_orders
  Objects: 6
  Total Size (MB): 0.61
  File Types: {'parquet': 1, 'avro': 2, 'json': 2, 'text': 1}
----------------------------------------------------------------------
[TABLE] fact_payments
  Objects: 6
  Total Size (MB): 0.31
  File Types: {'parquet': 1, 'avro': 2, 'json': 2, 'text': 1}
----------------------------------------------------------------------
[TABLE] fact_shipments
  Objects: 6
  Total Size (MB): 0.10
  File Types: {'parquet': 1, 'avro': 2, 'json': 2, 'text': 1}
----------------------------------------------------------------------
[INFO] Tot

## Data Quality Checks

In [14]:

# Function to get Parquet file schema from a table
def get_table_schema(table_name):
    """Get schema information from first Parquet file in table"""
    try:
        # Find data files (usually in data/* path)
        objects = minio_client.list_objects(
            BUCKET_NAME,
            prefix="warehouse/silver/" + table_name + "/data/",
            recursive=True
        )
        
        # Get first parquet file
        parquet_files = []
        for obj in objects:
            if obj.object_name.endswith('.parquet'):
                parquet_files.append(obj.object_name)
                if len(parquet_files) >= 1:
                    break
        
        if not parquet_files:
            return None, "No parquet files found"
        
        # Get file from MinIO and read schema
        response = minio_client.get_object(BUCKET_NAME, parquet_files[0])
        parquet_file = response.read()
        
        # Parse schema
        import io
        parquet_table = pq.read_table(io.BytesIO(parquet_file))
        schema = parquet_table.schema
        
        return schema, None
    except Exception as e:
        return None, str(e)

# Check schema for each table
print("\n[INFO] Checking table schemas...")
print("=" * 80)

for table_name in sorted(silver_tables.keys()):
    schema, error = get_table_schema(table_name)
    if error:
        print("[TABLE] " + table_name + " - ERROR: " + error)
    else:
        print("[TABLE] " + table_name)
        for field in schema:
            print("  Field: " + field.name + ", Type: " + str(field.type))
        print("-" * 80)



[INFO] Checking table schemas...
[TABLE] dim_customers
  Field: customer_sk, Type: string
  Field: user_id, Type: string
  Field: email, Type: string
  Field: country, Type: string
  Field: valid_from, Type: timestamp[us, tz=UTC]
  Field: valid_to, Type: timestamp[us, tz=UTC]
  Field: is_current, Type: bool
  Field: source_ts, Type: timestamp[us, tz=UTC]
  Field: updated_at, Type: timestamp[us, tz=UTC]
--------------------------------------------------------------------------------
[TABLE] dim_products
  Field: product_sk, Type: string
  Field: product_id, Type: string
  Field: title, Type: string
  Field: category, Type: string
  Field: price, Type: double
  Field: valid_from, Type: timestamp[us, tz=UTC]
  Field: valid_to, Type: timestamp[us, tz=UTC]
  Field: is_current, Type: bool
  Field: source_ts, Type: timestamp[us, tz=UTC]
  Field: updated_at, Type: timestamp[us, tz=UTC]
--------------------------------------------------------------------------------
[TABLE] fact_orders
  Field

In [15]:

# Function to count records in a table
def count_records_in_table(table_name):
    """Count total records by reading all parquet files"""
    try:
        total_records = 0
        file_count = 0
        
        objects = minio_client.list_objects(
            BUCKET_NAME,
            prefix="warehouse/silver/" + table_name + "/data/",
            recursive=True
        )
        
        for obj in objects:
            if obj.object_name.endswith('.parquet'):
                file_count += 1
                try:
                    response = minio_client.get_object(BUCKET_NAME, obj.object_name)
                    parquet_data = response.read()
                    
                    import io
                    parquet_table = pq.read_table(io.BytesIO(parquet_data))
                    total_records += len(parquet_table)
                except:
                    pass
        
        return total_records, file_count
    except Exception as e:
        return 0, 0

# Count records for each table
print("\n[INFO] Record count for each table:")
print("=" * 80)

table_stats = {}
for table_name in sorted(silver_tables.keys()):
    try:
        record_count, file_count = count_records_in_table(table_name)
        table_stats[table_name] = {'records': record_count, 'files': file_count}
        print("[TABLE] " + table_name)
        print("  Records: " + str(record_count))
        print("  Files: " + str(file_count))
        print("-" * 80)
    except Exception as e:
        print("[TABLE] " + table_name + " - ERROR: " + str(e))



[INFO] Record count for each table:
[TABLE] dim_customers
  Records: 2932
  Files: 2
--------------------------------------------------------------------------------
[TABLE] dim_products
  Records: 1789
  Files: 2
--------------------------------------------------------------------------------
[TABLE] fact_orders
  Records: 17653
  Files: 1
--------------------------------------------------------------------------------
[TABLE] fact_payments
  Records: 12162
  Files: 1
--------------------------------------------------------------------------------
[TABLE] fact_shipments
  Records: 4690
  Files: 1
--------------------------------------------------------------------------------


## SCD Type 2 Verification (dim_customers)

In [16]:

# Check SCD2 implementation for dim_customers
def verify_scd2_dim_customers():
    """Verify SCD2 columns exist: valid_from, valid_to, is_current"""
    try:
        schema, error = get_table_schema('dim_customers')
        if error:
            return False, error
        
        required_cols = ['valid_from', 'valid_to', 'is_current', 'user_id', 'email', 'country']
        available_cols = [field.name for field in schema]
        
        missing = [col for col in required_cols if col not in available_cols]
        
        if missing:
            return False, "Missing columns: " + str(missing)
        
        return True, "All SCD2 columns present"
    except Exception as e:
        return False, str(e)

scd2_ok, scd2_msg = verify_scd2_dim_customers()
print("\n[INFO] SCD2 Verification for dim_customers:")
print("  Status: " + ("VALID" if scd2_ok else "INVALID"))
print("  Message: " + scd2_msg)

# Check for records with is_current = false (historical records)
def check_scd2_history():
    """Check if SCD2 historical records exist"""
    try:
        objects = minio_client.list_objects(
            BUCKET_NAME,
            prefix="warehouse/silver/dim_customers/data/",
            recursive=True
        )
        
        historical_count = 0
        current_count = 0
        
        for obj in objects:
            if obj.object_name.endswith('.parquet'):
                try:
                    response = minio_client.get_object(BUCKET_NAME, obj.object_name)
                    parquet_data = response.read()
                    
                    import io
                    df = pq.read_table(io.BytesIO(parquet_data)).to_pandas()
                    
                    if 'is_current' in df.columns:
                        historical_count += len(df[df['is_current'] == False])
                        current_count += len(df[df['is_current'] == True])
                except:
                    pass
        
        return current_count, historical_count
    except Exception as e:
        return 0, 0

current_records, historical_records = check_scd2_history()
print("\n[INFO] SCD2 History Status:")
print("  Current Records (is_current=true): " + str(current_records))
print("  Historical Records (is_current=false): " + str(historical_records))
print("  History Tracking: " + ("ENABLED" if historical_records > 0 else "NO HISTORY YET"))



[INFO] SCD2 Verification for dim_customers:
  Status: VALID
  Message: All SCD2 columns present

[INFO] SCD2 History Status:
  Current Records (is_current=true): 500
  Historical Records (is_current=false): 2432
  History Tracking: ENABLED


In [18]:
import pyarrow.parquet as pq
import pandas as pd
import io

# In mẫu dữ liệu từ các bảng Silver
tables_to_explore = {
    'dim_customers': 3,
    'dim_products': 3,
    'fact_orders': 5,
    'fact_payments': 5,
    'fact_shipments': 3,
}

for table_name, sample_size in tables_to_explore.items():
    print(f"\n{'='*80}")
    print(f"{table_name.upper()}")
    print(f"{'='*80}")
    
    # Xây dựng đường dẫn
    table_path = f"warehouse/silver/{table_name}/"
    
    try:
        # Liệt kê các file parquet trong bảng (tìm data files)
        objects = list(minio_client.list_objects(BUCKET_NAME, table_path, recursive=True))
        parquet_files = [obj.object_name for obj in objects if obj.object_name.endswith('.parquet') and 'data/' in obj.object_name]
        
        if parquet_files:
            # Đọc file parquet đầu tiên
            file_path = parquet_files[0]
            print(f"\nReading: {file_path}")
            
            # Tải file từ MinIO
            response = minio_client.get_object(BUCKET_NAME, file_path)
            file_data = response.read()
            
            # Parse parquet
            parquet_table = pq.read_table(io.BytesIO(file_data))
            df = parquet_table.to_pandas()
            
            print(f"\nSchema ({len(df.columns)} columns):")
            for i, col in enumerate(df.columns, 1):
                print(f"  {i:2d}. {col:25s} | {str(df[col].dtype):15s}")
            
            print(f"\nSample {min(sample_size, len(df))} rows:")
            pd.set_option('display.max_columns', None)
            pd.set_option('display.width', 200)
            pd.set_option('display.max_colwidth', 30)
            print(df.head(sample_size).to_string())
            
            print(f"\nTotal rows in first file: {len(df)}")
        else:
            print(f"No parquet data files found in {table_path}")
            print(f"   Available files: {parquet_files[:3] if parquet_files else []}")
    except Exception as e:
        print(f"Error reading {table_name}: {str(e)}")


DIM_CUSTOMERS

Reading: warehouse/silver/dim_customers/data/valid_from_day=1970-01-01/00000-13-09621ee8-324b-43c4-bde4-eac469c18705-0-00002.parquet

Schema (9 columns):
   1. customer_sk               | object         
   2. user_id                   | object         
   3. email                     | object         
   4. country                   | object         
   5. valid_from                | datetime64[us, UTC]
   6. valid_to                  | datetime64[us, UTC]
   7. is_current                | bool           
   8. source_ts                 | datetime64[us, UTC]
   9. updated_at                | datetime64[us, UTC]

Sample 3 rows:
                                                        customer_sk       user_id                 email country                valid_from                         valid_to  is_current                        source_ts                       updated_at
0  c221f2f3adc4332006083e445c23c221620419686616e752464add525ede4f7c  usr_00jhoj50  user_322@example

In [11]:
def check_nulls_in_silver():
    silver_tables = [
        'dim_customers', 'dim_products', 
        'fact_orders', 'fact_payments', 'fact_shipments'
    ]
    
    print("📊 KIỂM TRA GIÁ TRỊ NULL TRONG TẦNG SILVER")
    print("=" * 60)
    
    for table in silver_tables:
        prefix = f"warehouse/silver/{table}/data/"
        try:
            # Lấy danh sách file parquet
            objects = list(minio_client.list_objects(BUCKET_NAME, prefix=prefix, recursive=True))
            parquet_files = [obj.object_name for obj in objects if obj.object_name.endswith('.parquet')]
            
            if not parquet_files:
                print(f"\n[TABLE] {table}: Không tìm thấy file dữ liệu.")
                continue
            
            # Đọc file gần nhất để kiểm tra (hoặc có thể loop qua tất cả nếu muốn)
            # Ở đây mình lọc file mới nhất để kiểm tra nhanh
            latest_file = sorted(parquet_files)[-1] 
            
            resp = minio_client.get_object(BUCKET_NAME, latest_file)
            df = pd.read_parquet(io.BytesIO(resp.read()))
            
            null_counts = df.isnull().sum()
            total_rows = len(df)
            
            print(f"\n[TABLE] {table} (Kiểm tra file: {latest_file.split('/')[-1]})")
            print(f"Tổng số dòng: {total_rows}")
            
            # Chỉ hiển thị các cột có NULL
            nulls_found = null_counts[null_counts > 0]
            if not nulls_found.empty:
                for col, count in nulls_found.items():
                    percentage = (count / total_rows) * 100
                    print(f"  - Cột '{col:<20}': {count:>5} dòng NULL ({percentage:>5.2f}%)")
            else:
                print("  ✅ Không có giá trị NULL nào.")
                
        except Exception as e:
            print(f"\n[ERROR] Lỗi khi kiểm tra bảng {table}: {e}")

# Thực hiện kiểm tra
check_nulls_in_silver()


📊 KIỂM TRA GIÁ TRỊ NULL TRONG TẦNG SILVER

[TABLE] dim_customers (Kiểm tra file: 00000-13-09621ee8-324b-43c4-bde4-eac469c18705-0-00001.parquet)
Tổng số dòng: 2432
  - Cột 'valid_to            ':   498 dòng NULL (20.48%)

[TABLE] dim_products (Kiểm tra file: 00000-27-81cf7eea-8e4f-4809-8cb8-ec694ba5615b-0-00001.parquet)
Tổng số dòng: 1589
  - Cột 'valid_to            ':   200 dòng NULL (12.59%)

[TABLE] fact_orders (Kiểm tra file: 00000-50-a53d2062-a022-41d1-89b4-f36dcda9ff44-0-00001.parquet)
Tổng số dòng: 17653
  - Cột 'payment_id          ':  5491 dòng NULL (31.11%)
  - Cột 'payment_method      ':  5491 dòng NULL (31.11%)
  - Cột 'payment_status      ':  5491 dòng NULL (31.11%)

[TABLE] fact_payments (Kiểm tra file: 00000-62-d5290c93-0cf7-4c6b-b27c-21262f294b10-0-00001.parquet)
Tổng số dòng: 12162
  ✅ Không có giá trị NULL nào.

[TABLE] fact_shipments (Kiểm tra file: 00000-71-37213fe7-b5cc-49c6-94b9-80356b2605da-0-00001.parquet)
Tổng số dòng: 4690
  ✅ Không có giá trị NULL nào.
